# Tests de mini-funcionalidades de filter_flows

Este notebook se usa para probar helpers y bloques internos de `filter_flows()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados

## Bloque 0. Preparación

### Imports generales

Qué hace: deja cargadas dependencias, helpers internos privados y estructuras necesarias para los tests.

In [1]:
from copy import deepcopy

import numpy as np
import pandas as pd
import h3

from pylondrina.datasets import FlowDataset
from pylondrina.errors import FilterError
from pylondrina.reports import Issue
from pylondrina.transforms.flows_filtering import (
    FlowFilterOptions,
    _normalize_filter_flows_request,
    _evaluate_where_mask_on_flows_df,
    _evaluate_h3_mask_on_flows_df,
    _resolve_filtered_flow_to_trips,
    _build_filter_flows_summary,
    _truncate_issues_with_limit,
    _build_issues_summary,
    _build_removed_rows_evidence,
    _build_metadata_out,
    _build_derived_flow_provenance,
    _extract_validated_flag,
    _normalize_where_clause,
    _resolve_flow_field_dtype,
    _allowed_ops_for_dtype,
    _validate_where_operator_value,
    _evaluate_where_operator_mask,
    _required_h3_fields_for_predicate,
    _coerce_datetime_scalar,
    _normalize_h3_value,
    _is_valid_h3_value,
    _sample_list,
    _is_empty_sequence,
    _to_json_serializable_or_none,
    _json_is_serializable,
    _json_safe_scalar,
    _utc_now_iso,
)

### Helpers de apoyo para test

Qué hace: construye fixtures canónicas pequeñas, contexto de request y aserciones útiles.

In [2]:
def h3_from_latlon(lat: float, lon: float, res: int = 8) -> str:
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, res)
    return h3.geo_to_h3(lat, lon, res)


def assert_issue_codes(
    issues: list[Issue],
    expected_present: list[str] | tuple[str, ...] = (),
    expected_absent: list[str] | tuple[str, ...] = (),
) -> None:
    codes = [issue.code for issue in issues]
    for code in expected_present:
        assert code in codes, f"Falta issue {code}. Codes emitidos: {codes}"
    for code in expected_absent:
        assert code not in codes, f"No debía aparecer {code}. Codes emitidos: {codes}"


def make_flowdataset_small() -> tuple[FlowDataset, dict]:
    origin_a = h3_from_latlon(-33.4500, -70.6500, 8)
    origin_b = h3_from_latlon(-33.4400, -70.6400, 8)
    origin_c = h3_from_latlon(-33.4600, -70.6600, 8)

    dest_a = h3_from_latlon(-33.4300, -70.6300, 8)
    dest_b = h3_from_latlon(-33.4700, -70.6200, 8)

    flows_df = pd.DataFrame(
        {
            "flow_id": ["f1", "f2", "f3", "f4"],
            "origin_h3_index": [origin_a, origin_a, origin_b, origin_c],
            "destination_h3_index": [dest_a, dest_b, dest_b, origin_a],
            "flow_count": [10, 5, 2, 1],
            "flow_value": [15.0, 5.0, 1.0, 3.5],
            "mode": ["bus", "metro", "walk", "bus"],
            "gender": ["F", "M", "F", None],
            "window_start_utc": pd.to_datetime(
                [
                    "2026-01-01T10:00:00Z",
                    "2026-01-01T12:00:00Z",
                    "2026-01-02T08:00:00Z",
                    "2026-01-03T09:00:00Z",
                ],
                utc=True,
            ),
            "window_end_utc": pd.to_datetime(
                [
                    "2026-01-01T10:30:00Z",
                    "2026-01-01T12:45:00Z",
                    "2026-01-02T08:15:00Z",
                    "2026-01-03T09:20:00Z",
                ],
                utc=True,
            ),
        }
    )

    flow_to_trips = pd.DataFrame(
        {
            "flow_id": ["f1", "f1", "f2", "f4"],
            "movement_id": ["m1", "m2", "m3", "m4"],
        }
    )

    metadata = {
        "dataset_id": "flows_demo",
        "artifact_id": "artifact_demo",
        "is_validated": False,
        "events": [
            {"op": "build_flows", "ts_utc": "2026-04-07T10:00:00Z"},
            {"op": "write_flows", "ts_utc": "2026-04-07T10:10:00Z"},
        ],
        "h3": {"resolution": 8},
    }

    aggregation_spec = {
        "h3_resolution": 8,
        "group_by": ["mode", "gender"],
    }

    provenance = {
        "source": "synthetic",
        "note": "fixture helper-level",
    }

    ds = FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips,
        aggregation_spec=aggregation_spec,
        source_trips=None,
        metadata=metadata,
        provenance=provenance,
    )

    cells = {
        "origin_a": origin_a,
        "origin_b": origin_b,
        "origin_c": origin_c,
        "dest_a": dest_a,
        "dest_b": dest_b,
    }

    return ds, cells


def make_request_ctx(
    flows: FlowDataset,
    *,
    strict: bool = False,
    where_provided: bool = True,
    h3_cells_provided: bool = False,
    spatial_predicate: str = "origin",
    keep_flow_to_trips: bool = True,
    keep_metadata: bool = True,
    max_issues: int = 1000,
) -> dict:
    md = flows.metadata
    return {
        "dataset_id": md.get("dataset_id"),
        "artifact_id": md.get("artifact_id"),
        "strict": strict,
        "where_provided": where_provided,
        "h3_cells_provided": h3_cells_provided,
        "spatial_predicate": spatial_predicate,
        "keep_flow_to_trips": keep_flow_to_trips,
        "keep_metadata": keep_metadata,
        "max_issues": max_issues,
    }

### Configuración visual

Qué hace: deja más legible la inspección manual cuando algo falle.

In [3]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports y fixtures OK")

Imports y fixtures OK


## Bloque 1. Utilidades internas de uso general

### Test 1.1 - _json_safe_scalar, _to_json_serializable_or_none, _json_is_serializable

Qué prueba: serialización estable para Issue.details, parameters y eventos.

In [4]:
ts = pd.Timestamp("2026-01-01T07:00:00Z")

assert _json_safe_scalar(None) is None
print(_json_safe_scalar(np.nan))
assert _json_safe_scalar(np.nan) is None
assert _json_safe_scalar(True) is True
assert _json_safe_scalar(ts).startswith("2026-01-01T07:00:00")

payload = {
    "a": 1,
    "b": [ts, np.nan, {"x": (1, 2), "y": None}],
}
normalized = _to_json_serializable_or_none(payload)

assert normalized["a"] == 1
assert normalized["b"][0].startswith("2026-01-01T07:00:00")
assert normalized["b"][1] is None
assert normalized["b"][2]["x"] == [1, 2]
assert _json_is_serializable(normalized) is True

print("Test 1.1 OK")

None
Test 1.1 OK


### Test 1.2 - _normalize_h3_value, _is_valid_h3_value, _required_h3_fields_for_predicate

Qué prueba: normalización básica de H3 y columnas requeridas por predicado espacial.

In [5]:
flows, cells = make_flowdataset_small()
valid_cell = cells["origin_a"]

assert _normalize_h3_value(valid_cell) == valid_cell
assert _normalize_h3_value(f"  {valid_cell}  ") == valid_cell
assert _normalize_h3_value("") is None
assert _normalize_h3_value("   ") is None
assert _normalize_h3_value(None) is None

assert _is_valid_h3_value(valid_cell) is True
assert _is_valid_h3_value("h3_invalido_total") is False

assert _required_h3_fields_for_predicate("origin") == ["origin_h3_index"]
assert _required_h3_fields_for_predicate("destination") == ["destination_h3_index"]
assert _required_h3_fields_for_predicate("both") == ["origin_h3_index", "destination_h3_index"]
assert _required_h3_fields_for_predicate("either") == ["origin_h3_index", "destination_h3_index"]

print("Test 1.2 OK")

Test 1.2 OK


### Test 1.3 - _resolve_flow_field_dtype y _allowed_ops_for_dtype

Qué prueba: detección del dtype lógico y matriz de operadores permitidos.

In [6]:
flows, _ = make_flowdataset_small()
df = flows.flows

assert _resolve_flow_field_dtype("flow_id", df["flow_id"]) == "string"
assert _resolve_flow_field_dtype("origin_h3_index", df["origin_h3_index"]) == "string"
assert _resolve_flow_field_dtype("flow_count", df["flow_count"]) == "int"
assert _resolve_flow_field_dtype("flow_value", df["flow_value"]) == "float"
assert _resolve_flow_field_dtype("window_start_utc", df["window_start_utc"]) == "datetime"
assert _resolve_flow_field_dtype("mode", df["mode"]) == "string"

ops_cat = _allowed_ops_for_dtype("categorical")
assert "eq" in ops_cat
assert "in" in ops_cat
assert "gt" not in ops_cat

ops_num = _allowed_ops_for_dtype("float")
assert "between" in ops_num
assert "gte" in ops_num

ops_bool = _allowed_ops_for_dtype("bool")
assert "eq" in ops_bool
assert "between" not in ops_bool

print("Test 1.3 OK")

Test 1.3 OK


### Test 1.4 - _validate_where_operator_value

Qué prueba: forma esperada del valor por operador.

In [7]:
ok, msg = _validate_where_operator_value("eq", "bus", "string")
assert ok is True

ok, msg = _validate_where_operator_value("in", ["bus", "metro"], "categorical")
assert ok is True

ok, msg = _validate_where_operator_value("in", [], "categorical")
assert ok is False

ok, msg = _validate_where_operator_value("gte", 3, "int")
assert ok is True

ok, msg = _validate_where_operator_value("gte", "3", "int")
assert ok is False

ok, msg = _validate_where_operator_value("between", [1, 10], "float")
assert ok is True

ok, msg = _validate_where_operator_value("between", [1], "float")
assert ok is False

ok, msg = _validate_where_operator_value("is_null", True, "string")
assert ok is True

ok, msg = _validate_where_operator_value("is_null", False, "string")
assert ok is False

print("Test 1.4 OK")

Test 1.4 OK


### Test 1.5 - _normalize_where_clause, _sample_list, _is_empty_sequence

Qué prueba: utilidades pequeñas del DSL y muestreo.

In [8]:
clause, shape = _normalize_where_clause("bus")
assert clause == {"eq": "bus"}
assert shape == "scalar"

clause, shape = _normalize_where_clause(["bus", "metro"])
assert clause == {"in": ["bus", "metro"]}
assert shape == "implicit_in"

clause, shape = _normalize_where_clause(("bus", "metro"))
assert clause == {"in": ["bus", "metro"]}
assert shape == "implicit_in"

clause, shape = _normalize_where_clause({"gte": 5})
assert clause == {"gte": 5}
assert shape == "mapping"

assert _is_empty_sequence([]) is True
assert _is_empty_sequence(()) is True
assert _is_empty_sequence(set()) is True
assert _is_empty_sequence("abc") is False

sample = _sample_list(["a", "b", "c", "d"], limit=2)
assert sample == ["a", "b"]

print("Test 1.5 OK")

Test 1.5 OK


### Test 1.6 - _coerce_datetime_scalar y _evaluate_where_operator_mask

Qué prueba: comparación numérica/datetime con máscaras booleanas.

In [9]:
flows, _ = make_flowdataset_small()
df = flows.flows

ts = _coerce_datetime_scalar("2026-01-01T08:00:00-03:00")
assert str(ts) == "2026-01-01 11:00:00+00:00"

mask_num = _evaluate_where_operator_mask(df["flow_count"], "int", "gte", 5)
assert mask_num.tolist() == [True, True, False, False]

mask_in = _evaluate_where_operator_mask(df["mode"], "string", "in", ["bus", "metro"])
assert mask_in.tolist() == [True, True, False, True]

mask_dt = _evaluate_where_operator_mask(
    df["window_start_utc"],
    "datetime",
    "between",
    ["2026-01-01T10:00:00Z", "2026-01-01T12:00:00Z"],
)
assert mask_dt.tolist() == [True, True, False, False]

mask_null = _evaluate_where_operator_mask(df["gender"], "string", "is_null", True)
assert mask_null.tolist() == [False, False, False, True]

print("Test 1.6 OK")

Test 1.6 OK


### Test 1.7 - _build_removed_rows_evidence

Qué prueba: muestra compacta de filas descartadas para Issue.details.

In [10]:
flows, _ = make_flowdataset_small()
removed_mask = pd.Series([False, True, True, False], index=flows.flows.index)

evidence = _build_removed_rows_evidence(
    flows.flows,
    removed_mask,
    value_fields=["mode", "flow_count"],
)

assert evidence["flow_id_sample_removed"] == ["f2", "f3"]
assert len(evidence["rows_sample_removed"]) == 2
assert evidence["rows_sample_removed"][0]["flow_id"] == "f2"
assert "mode" in evidence["rows_sample_removed"][0]
assert "flow_count" in evidence["rows_sample_removed"][0]

print("Test 1.7 OK")

Test 1.7 OK


### Test 1.8 - _build_metadata_out, _extract_validated_flag, _build_derived_flow_provenance, _utc_now_iso

Qué prueba: helpers de metadata/provenance que son críticos para trazabilidad.

In [11]:
flows, _ = make_flowdataset_small()

md_keep = _build_metadata_out(flows.metadata, keep_metadata=True)
md_drop = _build_metadata_out(flows.metadata, keep_metadata=False)

assert md_keep is not flows.metadata
assert "events" in md_keep
assert len(md_keep["events"]) == len(flows.metadata["events"])

assert "events" not in md_drop
assert md_drop["dataset_id"] == "flows_demo"
assert md_drop["artifact_id"] == "artifact_demo"

assert _extract_validated_flag(flows.metadata) is False
assert _extract_validated_flag({"flags": {"validated": True}}) is True
assert _extract_validated_flag({}) is False

prov = _build_derived_flow_provenance(flows)
assert prov is not None
assert "derived_from" in prov
assert prov["derived_from"][0]["dataset_id"] == "flows_demo"
assert "prior_events_summary" in prov
assert all(set(item.keys()) == {"op", "ts_utc"} for item in prov["prior_events_summary"])

ts_utc = _utc_now_iso()
assert ts_utc.endswith("Z")

print("Test 1.8 OK")

Test 1.8 OK


### Test 1.9 - _truncate_issues_with_limit y _build_issues_summary

Qué prueba: truncamiento y resumen por severidad/código.

In [12]:
issues_all = [
    Issue(level="info", code="X.INFO", message="i1"),
    Issue(level="warning", code="X.WARN", message="w1"),
    Issue(level="error", code="X.ERR", message="e1"),
    Issue(level="warning", code="X.WARN", message="w2"),
]

issues_truncated, limits = _truncate_issues_with_limit(issues_all, max_issues=3)

assert len(issues_truncated) == 3
assert issues_truncated[-1].code == "FLT_FLOW.REPORT.ISSUES_TRUNCATED"
assert limits is not None
assert limits["max_issues"] == 3
assert limits["issues_truncated"] is True

issues_summary = _build_issues_summary(issues_truncated)
assert issues_summary["counts"]["warning"] >= 1
assert issues_summary["counts"]["error"] >= 0
assert isinstance(issues_summary["top_codes"], list)

print("Test 1.9 OK")

Test 1.9 OK


## Bloque 2. Helper principal _normalize_filter_flows_request

### Test 2.1 - camino principal con options=None

Qué prueba: request efectivo por defecto, parameters, filters_requested y request_ctx.

In [13]:
flows, _ = make_flowdataset_small()
issues = []

options_eff, parameters, filters_requested, request_ctx = _normalize_filter_flows_request(
    flows,
    options=None,
    max_issues=1000,
    issues=issues,
)

assert isinstance(options_eff, FlowFilterOptions)
assert options_eff.where is None
assert options_eff.h3_cells is None
assert options_eff.spatial_predicate == "origin"

assert filters_requested == []
assert parameters["where"] is None
assert parameters["h3_cells"] is None
assert parameters["keep_flow_to_trips"] is True
assert request_ctx["dataset_id"] == "flows_demo"
assert request_ctx["artifact_id"] == "artifact_demo"
assert issues == []

print("Test 2.1 OK")

Test 2.1 OK


## Test 2.2 - normalización correcta de where + h3_cells deduplicado

Qué prueba: request explícito válido y deduplicación de H3.

In [14]:
flows, cells = make_flowdataset_small()
issues = []

options_in = FlowFilterOptions(
    where={"mode": ["bus", "metro"], "flow_count": {"gte": 2}},
    h3_cells=[cells["origin_a"], cells["origin_a"], cells["origin_b"]],
    spatial_predicate="either",
    keep_flow_to_trips=False,
    keep_metadata=False,
    strict=True,
)

options_eff, parameters, filters_requested, request_ctx = _normalize_filter_flows_request(
    flows,
    options=options_in,
    max_issues=50,
    issues=issues,
)

assert sorted(options_eff.h3_cells) == sorted({cells["origin_a"], cells["origin_b"]})
assert filters_requested == ["where", "h3_cells"]
assert parameters["spatial_predicate"] == "either"
assert parameters["keep_flow_to_trips"] is False
assert parameters["keep_metadata"] is False
assert request_ctx["strict"] is True
assert issues == []

print("Test 2.2 OK")

Test 2.2 OK


### Test 2.3 - fatal por max_issues inválido

Qué prueba: precheck fatal de configuración.

In [15]:
flows, _ = make_flowdataset_small()
issues = []

try:
    _normalize_filter_flows_request(
        flows,
        options=None,
        max_issues=0,
        issues=issues,
    )
    raise AssertionError("Debió levantar FilterError por max_issues inválido")
except FilterError as exc:
    assert exc.code == "FLT_FLOW.CONFIG.INVALID_MAX_ISSUES"

print("Test 2.3 OK")

Test 2.3 OK


### Test 2.4 - fatal por h3_cells vacío tras normalización

Qué prueba: h3_cells presente pero no interpretable como whitelist efectiva.

In [16]:
flows, _ = make_flowdataset_small()
issues = []

options_in = FlowFilterOptions(
    h3_cells=[None, "", "   ", np.nan],
)

try:
    _normalize_filter_flows_request(
        flows,
        options=options_in,
        max_issues=100,
        issues=issues,
    )
    raise AssertionError("Debió levantar FilterError por h3_cells vacío tras normalización")
except FilterError as exc:
    assert exc.code == "FLT_FLOW.CONFIG.H3_CELLS_EMPTY_AFTER_NORMALIZATION"

print("Test 2.4 OK")

Test 2.4 OK


### Test 2.5 - fatal por falta de columna canónica mínima

Qué prueba: contrato mínimo de FlowDataset.flows.

In [17]:
flows, _ = make_flowdataset_small()
flows_bad = deepcopy(flows)
flows_bad.flows = flows_bad.flows.drop(columns=["flow_value"])

issues = []

try:
    _normalize_filter_flows_request(
        flows_bad,
        options=None,
        max_issues=100,
        issues=issues,
    )
    raise AssertionError("Debió levantar FilterError por columnas canónicas faltantes")
except FilterError as exc:
    assert exc.code == "FLT_FLOW.CONTRACT.MISSING_CANONICAL_COLUMNS"

print("Test 2.5 OK")

Test 2.5 OK


### Test 2.6 - fatal por spatial_predicate inválido

Qué prueba: validación del predicado espacial antes de entrar a helpers de eje.

In [19]:
flows, _ = make_flowdataset_small()
issues = []

options_in = FlowFilterOptions(
    h3_cells=["dummy"],
    spatial_predicate="sideways",  # inválido
)

try:
    _normalize_filter_flows_request(
        flows,
        options=options_in,
        max_issues=100,
        issues=issues,
    )
    raise AssertionError("Debió levantar FilterError por spatial_predicate inválido")
except FilterError as exc:
    assert exc.code == "FLT_FLOW.CONFIG.INVALID_SPATIAL_PREDICATE"

print("Test 2.6 OK")

Test 2.6 OK


## Bloque 3. Helper principal _evaluate_where_mask_on_flows_df

### Test 3.1 - scalar implícito (eq) sobre categórico

Qué prueba: forma más básica del DSL.

In [20]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(flows, where_provided=True, h3_cells_provided=False)

mask, info = _evaluate_where_mask_on_flows_df(
    flows.flows,
    where={"mode": "bus"},
    issues=issues,
    request_ctx=ctx,
)

assert mask.tolist() == [True, False, False, True]
assert info["applied"] is True
assert info["fields_evaluated"] == ["mode"]
assert issues == []

print("Test 3.1 OK")

Test 3.1 OK


### Test 3.2 - secuencia implícita (in) + comparación numérica con AND entre campos

Qué prueba: combinación típica de filtros por atributos.

In [21]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(flows, where_provided=True, h3_cells_provided=False)

mask, info = _evaluate_where_mask_on_flows_df(
    flows.flows,
    where={
        "mode": ["bus", "metro"],
        "flow_count": {"gte": 5},
    },
    issues=issues,
    request_ctx=ctx,
)

assert mask.tolist() == [True, True, False, False]
assert info["applied"] is True
assert set(info["fields_evaluated"]) == {"mode", "flow_count"}
assert issues == []

print("Test 3.2 OK")

Test 3.2 OK


### Test 3.3 - between sobre datetime

Qué prueba: comparación temporal soportada por where en flows.

In [23]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(flows, where_provided=True, h3_cells_provided=False)

mask, info = _evaluate_where_mask_on_flows_df(
    flows.flows,
    where={
        "window_start_utc": {
            "between": ["2026-01-01T10:00:00Z", "2026-01-01T12:00:00Z"]
        }
    },
    issues=issues,
    request_ctx=ctx,
)

assert mask.tolist() == [True, True, False, False]
assert info["applied"] is True
assert info["fields_evaluated"] == ["window_start_utc"]
assert issues == []
print("Test 3.3 OK")

Test 3.3 OK


### Test 3.4 - campo inexistente -> issue y omisión de la regla

Qué prueba: degradación recuperable por campo faltante.

In [24]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(flows, where_provided=True, h3_cells_provided=False)

mask, info = _evaluate_where_mask_on_flows_df(
    flows.flows,
    where={"campo_inexistente": "x"},
    issues=issues,
    request_ctx=ctx,
)

assert mask is None
assert info["applied"] is False
assert_issue_codes(issues, expected_present=["FLT_FLOW.WHERE.FIELD_MISSING"])

print("Test 3.4 OK")

Test 3.4 OK


### Test 3.5 - operador incompatible con campo categórico

Qué prueba: error recuperable por matriz dtype ↔ operador.

In [25]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(flows, where_provided=True, h3_cells_provided=False)

mask, info = _evaluate_where_mask_on_flows_df(
    flows.flows,
    where={"mode": {"gte": 3}},
    issues=issues,
    request_ctx=ctx,
)

assert mask is None
assert info["applied"] is False
assert_issue_codes(issues, expected_present=["FLT_FLOW.WHERE.OPERATOR_INCOMPATIBLE"])

print("Test 3.5 OK")

Test 3.5 OK


### Test 3.6 - parseo datetime fallido

Qué prueba: degradación recuperable cuando el valor temporal no es interpretable.

In [26]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(flows, where_provided=True, h3_cells_provided=False)

mask, info = _evaluate_where_mask_on_flows_df(
    flows.flows,
    where={"window_start_utc": {"gte": "no_es_fecha"}},
    issues=issues,
    request_ctx=ctx,
)

assert mask is None
assert info["applied"] is False
assert_issue_codes(issues, expected_present=["FLT_FLOW.WHERE.DATETIME_PARSE_FAILED"])

print("Test 3.6 OK")

Test 3.6 OK


### Test 3.7 - mezcla de regla válida + regla inválida

Qué prueba: si una cláusula falla, el helper igual aplica las demás válidas.

In [27]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(flows, where_provided=True, h3_cells_provided=False)

mask, info = _evaluate_where_mask_on_flows_df(
    flows.flows,
    where={
        "mode": "bus",
        "campo_inexistente": "x",
    },
    issues=issues,
    request_ctx=ctx,
)

assert mask.tolist() == [True, False, False, True]
assert info["applied"] is True
assert info["fields_evaluated"] == ["mode"]
assert_issue_codes(issues, expected_present=["FLT_FLOW.WHERE.FIELD_MISSING"])

print("Test 3.7 OK")

Test 3.7 OK


## Bloque 4. Helper principal _evaluate_h3_mask_on_flows_df

### Test 4.1 - predicado origin

Qué prueba: filtro H3 básico sobre origen.

In [28]:
flows, cells = make_flowdataset_small()
issues = []
ctx = make_request_ctx(
    flows,
    where_provided=False,
    h3_cells_provided=True,
    spatial_predicate="origin",
)

mask, info = _evaluate_h3_mask_on_flows_df(
    flows.flows,
    h3_cells=[cells["origin_a"]],
    spatial_predicate="origin",
    issues=issues,
    request_ctx=ctx,
)

assert mask.tolist() == [True, True, False, False]
assert info["applied"] is True
assert info["valid_cells_count"] == 1
assert issues == []

print("Test 4.1 OK")

Test 4.1 OK


### Test 4.2 - predicado both

Qué prueba: ambos extremos deben pertenecer al conjunto.

In [29]:
flows, cells = make_flowdataset_small()
issues = []
ctx = make_request_ctx(
    flows,
    where_provided=False,
    h3_cells_provided=True,
    spatial_predicate="both",
)

mask, info = _evaluate_h3_mask_on_flows_df(
    flows.flows,
    h3_cells=[cells["origin_a"], cells["dest_a"]],
    spatial_predicate="both",
    issues=issues,
    request_ctx=ctx,
)

assert mask.tolist() == [True, False, False, False]
assert info["applied"] is True
assert issues == []

print("Test 4.2 OK")

Test 4.2 OK


### Test 4.3 - celdas inválidas degradan, pero las válidas se siguen usando

Qué prueba: warning/error agregado y continuidad del eje con las válidas.

In [30]:
flows, cells = make_flowdataset_small()
issues = []
ctx = make_request_ctx(
    flows,
    where_provided=False,
    h3_cells_provided=True,
    spatial_predicate="origin",
)

mask, info = _evaluate_h3_mask_on_flows_df(
    flows.flows,
    h3_cells=[cells["origin_a"], "celda_h3_invalida_total"],
    spatial_predicate="origin",
    issues=issues,
    request_ctx=ctx,
)

assert mask.tolist() == [True, True, False, False]
assert info["applied"] is True
assert_issue_codes(issues, expected_present=["FLT_FLOW.H3.INVALID_CELL_VALUES"])

print("Test 4.3 OK")

Test 4.3 OK


### Test 4.4 - faltan columnas H3 requeridas

Qué prueba: omisión recuperable del eje espacial cuando el dataframe no tiene columnas necesarias.

In [31]:
flows, cells = make_flowdataset_small()
flows_df_bad = flows.flows.drop(columns=["destination_h3_index"])

issues = []
ctx = make_request_ctx(
    flows,
    where_provided=False,
    h3_cells_provided=True,
    spatial_predicate="both",
)

mask, info = _evaluate_h3_mask_on_flows_df(
    flows_df_bad,
    h3_cells=[cells["origin_a"]],
    spatial_predicate="both",
    issues=issues,
    request_ctx=ctx,
)

assert mask is None
assert info["applied"] is False
assert_issue_codes(issues, expected_present=["FLT_FLOW.H3.COLUMNS_MISSING"])

print("Test 4.4 OK")

Test 4.4 OK


### Test 4.5 - todas las celdas inválidas

Qué prueba: el helper deja evidencia y no aplica el eje.

In [32]:
flows, _ = make_flowdataset_small()
issues = []
ctx = make_request_ctx(
    flows,
    where_provided=False,
    h3_cells_provided=True,
    spatial_predicate="origin",
)

mask, info = _evaluate_h3_mask_on_flows_df(
    flows.flows,
    h3_cells=["bad_1", "bad_2"],
    spatial_predicate="origin",
    issues=issues,
    request_ctx=ctx,
)

assert mask is None
assert info["applied"] is False
assert_issue_codes(issues, expected_present=["FLT_FLOW.H3.INVALID_CELL_VALUES"])

print("Test 4.5 OK")

Test 4.5 OK


## Bloque 5. Helper principal _resolve_filtered_flow_to_trips

### Test 5.1 - keep_flow_to_trips=False

Qué prueba: política explícita de no pedir auxiliar.

In [33]:
flows, _ = make_flowdataset_small()
issues = []

result, status = _resolve_filtered_flow_to_trips(
    flows.flow_to_trips,
    kept_flow_ids={"f1", "f2"},
    keep_flow_to_trips=False,
    issues=issues,
    request_ctx=make_request_ctx(flows, keep_flow_to_trips=False),
)

assert result is None
assert status == "not_requested"
assert issues == []

print("Test 5.1 OK")

Test 5.1 OK


### Test 5.2 - auxiliar solicitado pero ausente

Qué prueba: retorno None + issue informativo.

In [34]:
flows, _ = make_flowdataset_small()
issues = []

result, status = _resolve_filtered_flow_to_trips(
    None,
    kept_flow_ids={"f1", "f2"},
    keep_flow_to_trips=True,
    issues=issues,
    request_ctx=make_request_ctx(flows, keep_flow_to_trips=True),
)

assert result is None
assert status == "missing"
assert_issue_codes(issues, expected_present=["FLT_FLOW.AUX.FLOW_TO_TRIPS_REQUESTED_BUT_MISSING"])

print("Test 5.2 OK")

Test 5.2 OK


### Test 5.3 - auxiliar inválido por estructura incompleta

Qué prueba: descarte explícito del auxiliar.

In [35]:
flows, _ = make_flowdataset_small()
issues = []

bad_aux = pd.DataFrame(
    {
        "flow_id": ["f1", "f2"],
        # falta movement_id
    }
)

result, status = _resolve_filtered_flow_to_trips(
    bad_aux,
    kept_flow_ids={"f1"},
    keep_flow_to_trips=True,
    issues=issues,
    request_ctx=make_request_ctx(flows, keep_flow_to_trips=True),
)

assert result is None
assert status == "discarded_invalid"
assert_issue_codes(issues, expected_present=["FLT_FLOW.AUX.FLOW_TO_TRIPS_INVALID"])

print("Test 5.3 OK")

Test 5.3 OK


### Test 5.4 - sincronización correcta por flow_id

Qué prueba: happy path del auxiliar.

In [36]:
flows, _ = make_flowdataset_small()
issues = []

result, status = _resolve_filtered_flow_to_trips(
    flows.flow_to_trips,
    kept_flow_ids={"f1", "f4"},
    keep_flow_to_trips=True,
    issues=issues,
    request_ctx=make_request_ctx(flows, keep_flow_to_trips=True),
)

assert status == "synced"
assert result is not None
assert sorted(result["flow_id"].unique().tolist()) == ["f1", "f4"]
assert len(result) == 3
assert_issue_codes(issues, expected_present=["FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED"])

print("Test 5.4 OK")

Test 5.4 OK


## Bloque 6. Helper principal _build_filter_flows_summary

### Test 6.1 - summary mínimo estable

Qué prueba: estructura exacta del summary vigente.

In [37]:
summary = _build_filter_flows_summary(
    rows_in=10,
    rows_out=4,
    dropped_by_filter={"where": 4, "h3_cells": 2},
    filters_requested=["where", "h3_cells"],
    filters_applied=["where", "h3_cells"],
    filters_omitted=[],
    flow_to_trips_status="synced",
    limits=None,
)

assert summary["rows_in"] == 10
assert summary["rows_out"] == 4
assert summary["dropped_total"] == 6
assert summary["dropped_by_filter"] == {"where": 4, "h3_cells": 2}
assert summary["filters_requested"] == ["where", "h3_cells"]
assert summary["filters_applied"] == ["where", "h3_cells"]
assert summary["filters_omitted"] == []
assert summary["flow_to_trips_status"] == "synced"
assert "limits" not in summary

print("Test 6.1 OK")

Test 6.1 OK


### Test 6.2 - summary con bloque limits

Qué prueba: inclusión opcional del bloque de truncamiento.

In [38]:
summary = _build_filter_flows_summary(
    rows_in=10,
    rows_out=0,
    dropped_by_filter={"where": 6, "h3_cells": 4},
    filters_requested=["where", "h3_cells"],
    filters_applied=["where", "h3_cells"],
    filters_omitted=[],
    flow_to_trips_status="missing",
    limits={
        "max_issues": 3,
        "issues_truncated": True,
        "n_issues_emitted": 3,
        "n_issues_detected_total": 7,
    },
)

assert summary["limits"]["max_issues"] == 3
assert summary["limits"]["issues_truncated"] is True

print("Test 6.2 OK")

Test 6.2 OK


## Bloque 7: Smoke tests integrados de filter_flows

### Setup mínimo del bloque

Qué hace: agrega un helper pequeño para leer el último evento del dataset resultante.

In [40]:
from pylondrina.transforms.flows_filtering import filter_flows
def get_last_event(flows: FlowDataset) -> dict:
    events = flows.metadata.get("events", [])
    assert isinstance(events, list) and len(events) > 0, "No hay eventos en metadata['events']"
    return events[-1]

### Test 7.1 - Happy path mínimo integrado

Qué prueba: el camino feliz completo de la función pública: nuevo dataset, OperationReport, summary, parameters, evento filter_flows, sincronización de flow_to_trips, preservación de is_validated y no mutación del input.

In [42]:
flows, cells = make_flowdataset_small()
flows_before = flows.flows.copy(deep=True)
flow_to_trips_before = flows.flow_to_trips.copy(deep=True)
events_before = deepcopy(flows.metadata["events"])
validated_before = flows.metadata["is_validated"]

options = FlowFilterOptions(
    where={
        "mode": ["bus", "metro"],
        "flow_count": {"gte": 5},
    },
    h3_cells=[cells["origin_a"]],
    spatial_predicate="origin",
    keep_flow_to_trips=True,
    keep_metadata=True,
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
    max_issues=20,
)

# Resultado tabular
assert filtered is not flows
assert filtered.flows["flow_id"].tolist() == ["f1", "f2"]

# Reporte
assert report.ok is True
assert report.summary["rows_in"] == 4
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 2
assert report.summary["filters_requested"] == ["where", "h3_cells"]
assert report.summary["filters_applied"] == ["where", "h3_cells"]
assert report.summary["filters_omitted"] == []
assert report.summary["flow_to_trips_status"] == "synced"

# Parameters efectivos
assert report.parameters["max_issues"] == 20
assert report.parameters["spatial_predicate"] == "origin"
assert report.parameters["keep_flow_to_trips"] is True
assert report.parameters["keep_metadata"] is True
assert report.parameters["strict"] is False

# Metadata/evento
assert filtered.metadata["is_validated"] == validated_before
event = get_last_event(filtered)
assert event["op"] == "filter_flows"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

# flow_to_trips sincronizado
assert filtered.flow_to_trips is not None
assert filtered.flow_to_trips["flow_id"].tolist() == ["f1", "f1", "f2"]

# Input intacto
pd.testing.assert_frame_equal(flows.flows, flows_before)
pd.testing.assert_frame_equal(flows.flow_to_trips, flow_to_trips_before)
assert flows.metadata["events"] == events_before
assert flows.metadata["is_validated"] == validated_before

print("Flow dataset original")
display(flows.flows)
display(flows.flow_to_trips)

print("\nFlow dataset filtrado")
display(filtered.flows)
display(filtered.flow_to_trips)
display(report.issues)

Flow dataset original


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
2,f3,88b2c554cdfffff,88b2c5549dfffff,2,1.0,walk,F,2026-01-02 08:00:00+00:00,2026-01-02 08:15:00+00:00
3,f4,88b2c554e1fffff,88b2c554e9fffff,1,3.5,bus,None,2026-01-03 09:00:00+00:00,2026-01-03 09:20:00+00:00


,flow_id,movement_id
0,f1,m1
1,f1,m2
2,f2,m3
3,f4,m4



Flow dataset filtrado


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00


,flow_id,movement_id
0,f1,m1
1,f1,m2
2,f2,m3


[Issue(level='info', code='FLT_FLOW.WHERE.APPLIED', message='Se aplicó el eje `where` sobre FlowDataset.flows.', field=None, source_field=None, row_count=2, details={'dataset_id': 'flows_demo', 'artifact_id': 'artifact_demo', 'strict': False, 'where_provided': True, 'h3_cells_provided': True, 'spatial_predicate': 'origin', 'keep_flow_to_trips': True, 'keep_metadata': True, 'max_issues': 20, 'n_flows_in': 4, 'n_flows_out': 2, 'rows_in': 4, 'rows_out': 2, 'dropped_total': 2, 'filters_requested': ['where', 'h3_cells'], 'filters_applied': ['where', 'h3_cells'], 'filters_omitted': [], 'dropped_by_filter': 2, 'flow_id_sample_removed': ['f3', 'f4'], 'rows_sample_removed': [{'flow_id': 'f3', 'mode': 'walk', 'flow_count': 2}, {'flow_id': 'f4', 'mode': 'bus', 'flow_count': 1}], 'reason': 'where_applied', 'action': 'apply_where_axis', 'fields_evaluated': ['mode', 'flow_count'], 'rules_evaluated': 2}),
 Issue(level='info', code='FLT_FLOW.H3.APPLIED', message='Se aplicó el eje `h3_cells` sobre `ori

### Test 7.2 - Filtro simple con keep_metadata=False

Qué prueba: que el filtrado siga funcionando bien sin copiar historial de eventos ni agregar evento nuevo, pero preservando metadata operativa mínima.

In [49]:
flows, _ = make_flowdataset_small()
flows_before = flows.flows.copy(deep=True)
events_before = deepcopy(flows.metadata["events"])
validated_before = flows.metadata["is_validated"]

options = FlowFilterOptions(
    where={"mode": "bus"},
    keep_metadata=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
)

# Subconjunto esperado
assert filtered.flows["flow_id"].tolist() == ["f1", "f4"]

# Reporte
assert report.ok is True
assert report.summary["rows_in"] == 4
assert report.summary["rows_out"] == 2
assert report.summary["filters_requested"] == ["where"]
assert report.summary["filters_applied"] == ["where"]
assert report.summary["filters_omitted"] == []

# No debe haber historial de events ni evento nuevo
assert "events" not in filtered.metadata

# Metadata mínima preservada
assert filtered.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert filtered.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert filtered.metadata["is_validated"] == validated_before
assert filtered.metadata["h3"] == flows.metadata["h3"]

# Input intacto
pd.testing.assert_frame_equal(flows.flows, flows_before)
assert flows.metadata["events"] == events_before
assert flows.metadata["is_validated"] == validated_before

print("Flow dataset original")
display(flows.flows)
display(flows.flow_to_trips)

print("\nFlow dataset filtrado")
display(filtered.flows)
display(filtered.flow_to_trips)
display(report.issues)

Flow dataset original


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
2,f3,88b2c554cdfffff,88b2c5549dfffff,2,1.0,walk,F,2026-01-02 08:00:00+00:00,2026-01-02 08:15:00+00:00
3,f4,88b2c554e1fffff,88b2c554e9fffff,1,3.5,bus,None,2026-01-03 09:00:00+00:00,2026-01-03 09:20:00+00:00


,flow_id,movement_id
0,f1,m1
1,f1,m2
2,f2,m3
3,f4,m4



Flow dataset filtrado


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
3,f4,88b2c554e1fffff,88b2c554e9fffff,1,3.5,bus,None,2026-01-03 09:00:00+00:00,2026-01-03 09:20:00+00:00


,flow_id,movement_id
0,f1,m1
1,f1,m2
3,f4,m4


[Issue(level='info', code='FLT_FLOW.WHERE.APPLIED', message='Se aplicó el eje `where` sobre FlowDataset.flows.', field=None, source_field=None, row_count=2, details={'dataset_id': 'flows_demo', 'artifact_id': 'artifact_demo', 'strict': False, 'where_provided': True, 'h3_cells_provided': False, 'spatial_predicate': 'origin', 'keep_flow_to_trips': True, 'keep_metadata': False, 'max_issues': 1000, 'n_flows_in': 4, 'n_flows_out': 2, 'rows_in': 4, 'rows_out': 2, 'dropped_total': 2, 'filters_requested': ['where'], 'filters_applied': ['where'], 'filters_omitted': [], 'dropped_by_filter': 2, 'flow_id_sample_removed': ['f2', 'f3'], 'rows_sample_removed': [{'flow_id': 'f2', 'mode': 'metro'}, {'flow_id': 'f3', 'mode': 'walk'}], 'reason': 'where_applied', 'action': 'apply_where_axis', 'fields_evaluated': ['mode'], 'rules_evaluated': 1}),
 Issue(level='info', code='FLT_FLOW.AUX.FLOW_TO_TRIPS_SYNCED', message='Se filtró `flow_to_trips` para mantener consistencia con los `flow_id` retenidos.', field=

### Test 7.3 - Degradado no estricto: where inválido pero H3 sí aplicable

Qué prueba: ruta recuperable con strict=False: el eje where se omite con issue error, el eje H3 sí se aplica, el resultado retorna dataset + reporte + evento.

In [50]:
flows, cells = make_flowdataset_small()
flows_before = flows.flows.copy(deep=True)
events_before = deepcopy(flows.metadata["events"])
validated_before = flows.metadata["is_validated"]

options = FlowFilterOptions(
    where={"campo_inexistente": "x"},
    h3_cells=[cells["origin_a"]],
    spatial_predicate="origin",
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
)

codes = [issue.code for issue in report.issues]

# where se omite, H3 sí se aplica
assert "FLT_FLOW.WHERE.FIELD_MISSING" in codes
assert "FLT_FLOW.H3.APPLIED" in codes

assert report.ok is False
assert report.summary["filters_requested"] == ["where", "h3_cells"]
assert report.summary["filters_applied"] == ["h3_cells"]
assert report.summary["filters_omitted"] == ["where"]

# origin == origin_a conserva f1, f2
assert filtered.flows["flow_id"].tolist() == ["f1", "f2"]

# Se preserva is_validated
assert filtered.metadata["is_validated"] == validated_before

# Se registra evento en el resultado
event = get_last_event(filtered)
assert event["op"] == "filter_flows"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

# Input intacto
pd.testing.assert_frame_equal(flows.flows, flows_before)
assert flows.metadata["events"] == events_before
assert flows.metadata["is_validated"] == validated_before

print("Flow dataset original")
display(flows.flows)
display(flows.flow_to_trips)

print("\nFlow dataset filtrado")
display(filtered.flows)
display(filtered.flow_to_trips)
display(report.issues)

Flow dataset original


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
2,f3,88b2c554cdfffff,88b2c5549dfffff,2,1.0,walk,F,2026-01-02 08:00:00+00:00,2026-01-02 08:15:00+00:00
3,f4,88b2c554e1fffff,88b2c554e9fffff,1,3.5,bus,None,2026-01-03 09:00:00+00:00,2026-01-03 09:20:00+00:00


,flow_id,movement_id
0,f1,m1
1,f1,m2
2,f2,m3
3,f4,m4



Flow dataset filtrado


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00


,flow_id,movement_id
0,f1,m1
1,f1,m2
2,f2,m3


[Issue(level='error', code='FLT_FLOW.WHERE.FIELD_MISSING', message="El eje `where` no puede evaluar el campo 'campo_inexistente' porque no existe en FlowDataset.flows; esa regla se omitirá.", field='campo_inexistente', source_field=None, row_count=4, details={'dataset_id': 'flows_demo', 'artifact_id': 'artifact_demo', 'strict': False, 'where_provided': True, 'h3_cells_provided': True, 'spatial_predicate': 'origin', 'keep_flow_to_trips': True, 'keep_metadata': True, 'max_issues': 1000, 'n_flows_in': 4, 'n_flows_out': 4, 'rows_in': 4, 'rows_out': 4, 'dropped_total': 0, 'field': 'campo_inexistente', 'operator': None, 'value': None, 'value_repr': None, 'expected_type': 'existing column in FlowDataset.flows', 'dtype_observed': None, 'available_fields_sample': ['flow_id', 'origin_h3_index', 'destination_h3_index', 'flow_count', 'flow_value', 'mode', 'gender', 'window_start_utc', 'window_end_utc'], 'available_fields_total': 9, 'reason': 'field_missing', 'action': 'omit_where_rule'}),
 Issue(l

### Test 7.4 - strict=True sobre error recuperable

Qué prueba: que un error recuperable por eje escale a FilterError cuando strict=True, sin mutar el input.

In [45]:
flows, cells = make_flowdataset_small()
flows_before = flows.flows.copy(deep=True)
flow_to_trips_before = flows.flow_to_trips.copy(deep=True)
events_before = deepcopy(flows.metadata["events"])
validated_before = flows.metadata["is_validated"]

raised = None
try:
    filter_flows(
        flows,
        options=FlowFilterOptions(
            where={"campo_inexistente": "x"},
            h3_cells=[cells["origin_a"]],
            spatial_predicate="origin",
            strict=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FilterError)
assert raised.issue is not None
assert raised.issue.code == "FLT_FLOW.WHERE.FIELD_MISSING"
assert raised.issues is not None
assert any(issue.code == "FLT_FLOW.WHERE.FIELD_MISSING" for issue in raised.issues)

# Input intacto
pd.testing.assert_frame_equal(flows.flows, flows_before)
pd.testing.assert_frame_equal(flows.flow_to_trips, flow_to_trips_before)
assert flows.metadata["events"] == events_before
assert flows.metadata["is_validated"] == validated_before

display(raised.issues)

(Issue(level='error', code='FLT_FLOW.WHERE.FIELD_MISSING', message="El eje `where` no puede evaluar el campo 'campo_inexistente' porque no existe en FlowDataset.flows; esa regla se omitirá.", field='campo_inexistente', source_field=None, row_count=4, details={'dataset_id': 'flows_demo', 'artifact_id': 'artifact_demo', 'strict': True, 'where_provided': True, 'h3_cells_provided': True, 'spatial_predicate': 'origin', 'keep_flow_to_trips': True, 'keep_metadata': True, 'max_issues': 1000, 'n_flows_in': 4, 'n_flows_out': 4, 'rows_in': 4, 'rows_out': 4, 'dropped_total': 0, 'field': 'campo_inexistente', 'operator': None, 'value': None, 'value_repr': None, 'expected_type': 'existing column in FlowDataset.flows', 'dtype_observed': None, 'available_fields_sample': ['flow_id', 'origin_h3_index', 'destination_h3_index', 'flow_count', 'flow_value', 'mode', 'gender', 'window_start_utc', 'window_end_utc'], 'available_fields_total': 9, 'reason': 'field_missing', 'action': 'omit_where_rule'}),
 Issue(le

### Test 7.5 - Resultado vacío retornable

Qué prueba: caso donde el filtro se aplica correctamente pero deja el dataset vacío; debe retornar warning y seguir siendo ruta normal.

In [46]:
flows, _ = make_flowdataset_small()
flows_before = flows.flows.copy(deep=True)
events_before = deepcopy(flows.metadata["events"])
validated_before = flows.metadata["is_validated"]

options = FlowFilterOptions(
    where={
        "mode": "bus",
        "flow_count": {"lt": 0},
    },
)

filtered, report = filter_flows(
    flows,
    options=options,
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert "FLT_FLOW.RESULT.EMPTY_DATASET" in codes

assert report.summary["rows_in"] == 4
assert report.summary["rows_out"] == 0
assert report.summary["dropped_total"] == 4
assert filtered.flows.empty is True

event = get_last_event(filtered)
assert event["op"] == "filter_flows"
assert event["summary"] == report.summary

assert filtered.metadata["is_validated"] == validated_before

# Input intacto
pd.testing.assert_frame_equal(flows.flows, flows_before)
assert flows.metadata["events"] == events_before

display(flows.flows)
display(filtered.flows)
display(report.issues)

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
2,f3,88b2c554cdfffff,88b2c5549dfffff,2,1.0,walk,F,2026-01-02 08:00:00+00:00,2026-01-02 08:15:00+00:00
3,f4,88b2c554e1fffff,88b2c554e9fffff,1,3.5,bus,None,2026-01-03 09:00:00+00:00,2026-01-03 09:20:00+00:00


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc


[Issue(level='info', code='FLT_FLOW.WHERE.APPLIED', message='Se aplicó el eje `where` sobre FlowDataset.flows.', field=None, source_field=None, row_count=4, details={'dataset_id': 'flows_demo', 'artifact_id': 'artifact_demo', 'strict': False, 'where_provided': True, 'h3_cells_provided': False, 'spatial_predicate': 'origin', 'keep_flow_to_trips': True, 'keep_metadata': True, 'max_issues': 1000, 'n_flows_in': 4, 'n_flows_out': 0, 'rows_in': 4, 'rows_out': 0, 'dropped_total': 4, 'filters_requested': ['where'], 'filters_applied': ['where'], 'filters_omitted': [], 'dropped_by_filter': 4, 'flow_id_sample_removed': ['f1', 'f2', 'f3', 'f4'], 'rows_sample_removed': [{'flow_id': 'f1', 'mode': 'bus', 'flow_count': 10}, {'flow_id': 'f2', 'mode': 'metro', 'flow_count': 5}, {'flow_id': 'f3', 'mode': 'walk', 'flow_count': 2}, {'flow_id': 'f4', 'mode': 'bus', 'flow_count': 1}], 'reason': 'where_applied', 'action': 'apply_where_axis', 'fields_evaluated': ['mode', 'flow_count'], 'rules_evaluated': 2}),


### Test 7.6 - Truncamiento de issues con max_issues

Qué prueba: que el truncamiento funcione a nivel de la función pública y quede reflejado en summary y en el evento.

In [47]:
flows, _ = make_flowdataset_small()

options = FlowFilterOptions(
    where={
        "does_not_exist": "x",                   # FIELD_MISSING
        "mode": {"gt": "bus"},                   # OPERATOR_INCOMPATIBLE
        "window_start_utc": {"gte": "bad_ts"},   # DATETIME_PARSE_FAILED
        "gender": {"is_null": False},            # INVALID_VALUE_SHAPE
    },
    strict=False,
)

filtered, report = filter_flows(
    flows,
    options=options,
    max_issues=2,
)

codes = [issue.code for issue in report.issues]

assert "FLT_FLOW.REPORT.ISSUES_TRUNCATED" in codes
assert report.summary["limits"]["issues_truncated"] is True
assert report.summary["limits"]["max_issues"] == 2
assert report.summary["limits"]["n_issues_emitted"] <= 2
assert report.summary["limits"]["n_issues_detected_total"] >= report.summary["limits"]["n_issues_emitted"]

event = get_last_event(filtered)
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

display(filtered.flows)
display(report.issues)

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00
2,f3,88b2c554cdfffff,88b2c5549dfffff,2,1.0,walk,F,2026-01-02 08:00:00+00:00,2026-01-02 08:15:00+00:00
3,f4,88b2c554e1fffff,88b2c554e9fffff,1,3.5,bus,None,2026-01-03 09:00:00+00:00,2026-01-03 09:20:00+00:00


[Issue(level='error', code='FLT_FLOW.WHERE.FIELD_MISSING', message="El eje `where` no puede evaluar el campo 'does_not_exist' porque no existe en FlowDataset.flows; esa regla se omitirá.", field='does_not_exist', source_field=None, row_count=4, details={'dataset_id': 'flows_demo', 'artifact_id': 'artifact_demo', 'strict': False, 'where_provided': True, 'h3_cells_provided': False, 'spatial_predicate': 'origin', 'keep_flow_to_trips': True, 'keep_metadata': True, 'max_issues': 2, 'n_flows_in': 4, 'n_flows_out': 4, 'rows_in': 4, 'rows_out': 4, 'dropped_total': 0, 'field': 'does_not_exist', 'operator': None, 'value': None, 'value_repr': None, 'expected_type': 'existing column in FlowDataset.flows', 'dtype_observed': None, 'available_fields_sample': ['flow_id', 'origin_h3_index', 'destination_h3_index', 'flow_count', 'flow_value', 'mode', 'gender', 'window_start_utc', 'window_end_utc'], 'available_fields_total': 9, 'reason': 'field_missing', 'action': 'omit_where_rule'}),
 Issue(level='warni

### Test 7.7 - Política de flow_to_trips ausente pero solicitada

Qué prueba: que la función pública degrade correctamente cuando el auxiliar no existe, sin romper el filtrado principal.

In [48]:
flows, cells = make_flowdataset_small()
flows_no_aux = FlowDataset(
    flows=flows.flows.copy(deep=True),
    flow_to_trips=None,
    aggregation_spec=deepcopy(flows.aggregation_spec),
    source_trips=flows.source_trips,
    metadata=deepcopy(flows.metadata),
    provenance=deepcopy(flows.provenance),
)

options = FlowFilterOptions(
    h3_cells=[cells["origin_a"]],
    spatial_predicate="origin",
    keep_flow_to_trips=True,
)

filtered, report = filter_flows(
    flows_no_aux,
    options=options,
)

codes = [issue.code for issue in report.issues]

assert "FLT_FLOW.AUX.FLOW_TO_TRIPS_REQUESTED_BUT_MISSING" in codes
assert report.summary["flow_to_trips_status"] == "missing"
assert filtered.flow_to_trips is None

# El filtrado principal sí ocurre
assert filtered.flows["flow_id"].tolist() == ["f1", "f2"]

event = get_last_event(filtered)
assert event["summary"] == report.summary

display(filtered.flows)
display(report.issues)

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,gender,window_start_utc,window_end_utc
0,f1,88b2c554e9fffff,88b2c556a1fffff,10,15.0,bus,F,2026-01-01 10:00:00+00:00,2026-01-01 10:30:00+00:00
1,f2,88b2c554e9fffff,88b2c5549dfffff,5,5.0,metro,M,2026-01-01 12:00:00+00:00,2026-01-01 12:45:00+00:00


[Issue(level='info', code='FLT_FLOW.H3.APPLIED', message='Se aplicó el eje `h3_cells` sobre `origin_h3_index` / `destination_h3_index`.', field=None, source_field=None, row_count=2, details={'dataset_id': 'flows_demo', 'artifact_id': 'artifact_demo', 'strict': False, 'where_provided': False, 'h3_cells_provided': True, 'spatial_predicate': 'origin', 'keep_flow_to_trips': True, 'keep_metadata': True, 'max_issues': 1000, 'n_flows_in': 4, 'n_flows_out': 2, 'rows_in': 4, 'rows_out': 2, 'dropped_total': 2, 'filters_requested': ['h3_cells'], 'filters_applied': ['h3_cells'], 'filters_omitted': [], 'dropped_by_filter': 2, 'flow_id_sample_removed': ['f3', 'f4'], 'rows_sample_removed': [{'flow_id': 'f3', 'origin_h3_index': '88b2c554cdfffff', 'destination_h3_index': '88b2c5549dfffff'}, {'flow_id': 'f4', 'origin_h3_index': '88b2c554e1fffff', 'destination_h3_index': '88b2c554e9fffff'}], 'reason': 'h3_axis_applied', 'action': 'apply_h3_axis', 'h3_cells_count': 1}),
 Issue(level='info', code='FLT_FLOW